# 💊 AI Medication Management Agent
### Architecture: INPUT → LSTM (Deep Learning) → GPT-4 Agent Brain → Tool → Output

This agent:
1. Takes a patient's recent dose history as input
2. Uses an **LSTM deep learning model** to predict whether the next dose will be missed
3. Passes that prediction + **memory of past interactions** to **GPT-4** to decide what action to take
4. Triggers a **notification tool** based on that decision

**Deep Learning Concepts Used:**
- LSTM (Long Short-Term Memory) for sequence prediction
- Transformer-based LLM (GPT-4) for reasoning
- Conversation memory for context-aware decisions

---
## Cell 1 — Install & Import

In [1]:
# Install required libraries (run once in Colab)
!pip install tensorflow numpy pandas openai --quiet
print('✅ Libraries installed')

✅ Libraries installed


In [2]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from openai import OpenAI
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries loaded successfully')

✅ Libraries loaded successfully


---
## Cell 2 — API Key Setup

Paste your OpenAI API key below. In a real deployment this would come from an environment variable (see `.env.example`).

In [3]:

client = OpenAI(api_key=OPENAI_API_KEY)
print('✅ OpenAI client ready')

✅ OpenAI client ready


---
## Cell 3 — Patient Data

Simulating a patient's medication log: `1 = dose taken`, `0 = dose missed`

In [4]:
# Simulated patient medication history (1 = taken, 0 = missed)
# 12 scheduled doses over 6 days (morning + evening)
dose_history = [1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0]

df = pd.DataFrame({
    'Day':    ['Mon AM','Mon PM','Tue AM','Tue PM','Wed AM','Wed PM',
               'Thu AM','Thu PM','Fri AM','Fri PM','Sat AM','Sat PM'],
    'Taken':  dose_history,
    'Status': ['✅ Taken' if x == 1 else '❌ Missed' for x in dose_history]
})

total = len(dose_history)
taken = sum(dose_history)
pct   = (taken / total) * 100

print('=== PATIENT MEDICATION LOG ===')
print(df.to_string(index=False))
print(f'\n📊 Adherence Rate: {taken}/{total} doses taken ({pct:.1f}%)')
if pct < 70:
    print('⚠️  WARNING: Adherence below recommended 70% threshold')

=== PATIENT MEDICATION LOG ===
   Day  Taken   Status
Mon AM      1  ✅ Taken
Mon PM      1  ✅ Taken
Tue AM      0 ❌ Missed
Tue PM      1  ✅ Taken
Wed AM      0 ❌ Missed
Wed PM      0 ❌ Missed
Thu AM      1  ✅ Taken
Thu PM      1  ✅ Taken
Fri AM      0 ❌ Missed
Fri PM      1  ✅ Taken
Sat AM      0 ❌ Missed
Sat PM      0 ❌ Missed

📊 Adherence Rate: 6/12 doses taken (50.0%)
⚠️  WARNING: Adherence below recommended 70% threshold


---
## Cell 4 — Prepare Sequences for LSTM

The LSTM learns from **sequences of 3 past doses** to predict whether the next dose will be taken.

> **Why LSTM?** A rule-based system can only say "if missed 2 in a row, send alert." An LSTM learns *temporal patterns* specific to each patient — a key deep learning advantage.

In [5]:
SEQ_LEN = 3

def create_sequences(data, seq_length=3):
    """Slide a window across the dose history to build training pairs."""
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i : i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

data = np.array(dose_history, dtype=np.float32)
X, y = create_sequences(data, SEQ_LEN)
X = X.reshape((X.shape[0], X.shape[1], 1))  # (samples, timesteps, features)

print(f'Training samples: {X.shape[0]}')
print(f'Each sample = {SEQ_LEN} past doses → predict next dose')
print(f'Example — Input: {X[0].flatten()} → Label: {int(y[0])} ({"Taken" if y[0]==1 else "Missed"})')

Training samples: 9
Each sample = 3 past doses → predict next dose
Example — Input: [1. 1. 0.] → Label: 1 (Taken)


---
## Cell 5 — Build & Train the LSTM Model

This is the **Deep Learning component**. The LSTM uses gated memory cells to retain information across time steps — ideal for sequential dose data.

In [6]:
model = Sequential([
    LSTM(50, activation='relu', input_shape=(SEQ_LEN, 1)),
    Dense(1, activation='sigmoid')  # Output: probability dose will be TAKEN
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print('=== LSTM MODEL ARCHITECTURE ===')
model.summary()

print('\n⏳ Training...')
history = model.fit(X, y, epochs=50, verbose=0)

final_acc  = history.history['accuracy'][-1]
final_loss = history.history['loss'][-1]
print(f'✅ Training complete — Loss: {final_loss:.4f} | Accuracy: {final_acc:.1%}')

=== LSTM MODEL ARCHITECTURE ===


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50)             │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,451 (40.82 KB)

 Trainable params: 10,451 (40.82 KB)

 Non-trainable params: 0 (0.00 B)


⏳ Training...
✅ Training complete — Loss: 0.6676 | Accuracy: 55.6%


---
## Cell 6 — Memory System

The agent keeps a **conversation memory** so GPT-4 can reason across multiple patient interactions, not just one at a time. This is the memory component required by the assignment.

In [7]:
# Memory: stores the conversation history sent to GPT-4
# Each entry is a {role, content} dict — standard OpenAI chat format
conversation_memory = [
    {
        "role": "system",
        "content": (
            "You are a clinical medication adherence assistant. "
            "You receive a patient's dose history and an LSTM model prediction, "
            "then decide the best action to take. "
            "Always respond in this exact format:\n"
            "ACTION: <Send Reminder | Escalate to Doctor | No Action>\n"
            "REASON: <one sentence explaining why>"
        )
    }
]

def add_to_memory(role, content):
    """Append a message to conversation memory."""
    conversation_memory.append({"role": role, "content": content})

print(f'✅ Memory initialized with system prompt')
print(f'   Memory currently holds {len(conversation_memory)} message(s)')

✅ Memory initialized with system prompt
   Memory currently holds 1 message(s)


---
## Cell 7 — Define Pipeline Components

```
[INPUT] → [LSTM] → [GPT-4 AGENT BRAIN + MEMORY] → [TOOL] → [OUTPUT]
```

In [8]:
# ─────────────────────────────────────────────
# COMPONENT 1: DEEP LEARNING MODEL (LSTM)
# ─────────────────────────────────────────────
def lstm_predict(sequence):
    """Use the trained LSTM to predict the next dose outcome."""
    arr  = np.array(sequence, dtype=np.float32).reshape((1, SEQ_LEN, 1))
    prob = model.predict(arr, verbose=0)[0][0]
    label = 'Taken' if prob > 0.5 else 'Missed'
    return label, prob


# ─────────────────────────────────────────────
# COMPONENT 2: AGENT BRAIN — Real GPT-4 Call
# Uses conversation memory for context-aware reasoning
# ─────────────────────────────────────────────
def agent_brain(sequence, prediction, confidence, adherence_pct):
    """Send context to GPT-4 and get a reasoned action decision."""
    user_message = (
        f"Patient data:\n"
        f"- Last 3 doses: {sequence}\n"
        f"- LSTM prediction for next dose: {prediction} (confidence: {confidence:.1%})\n"
        f"- Overall weekly adherence: {adherence_pct:.1f}%\n"
        f"\nWhat action should be taken?"
    )

    # Add patient context to memory
    add_to_memory("user", user_message)

    # Call GPT-4 with full conversation history
    response = client.chat.completions.create(
        model="gpt-4",
        messages=conversation_memory,
        temperature=0.2   # Low temp = more consistent clinical decisions
    )

    reply = response.choices[0].message.content

    # Save GPT-4's response to memory for future context
    add_to_memory("assistant", reply)

    # Parse the action from the structured response
    action = 'No Action'
    for line in reply.split('\n'):
        if line.startswith('ACTION:'):
            action = line.replace('ACTION:', '').strip()
    reason = ''
    for line in reply.split('\n'):
        if line.startswith('REASON:'):
            reason = line.replace('REASON:', '').strip()

    return action, reason


# ─────────────────────────────────────────────
# COMPONENT 3: NOTIFICATION TOOL
# ─────────────────────────────────────────────
def notification_tool(action):
    """Execute the action decided by the agent brain."""
    if action == 'Send Reminder':
        print('  🔔 ALERT: Sending medication reminder to patient')
        print('  📱 SMS dispatched to patient phone')
        print('  👩‍⚕️  Caregiver dashboard updated')
    elif action == 'Escalate to Doctor':
        print('  🚨 ESCALATION: Notifying physician of non-adherence pattern')
        print('  📋 Patient chart flagged for follow-up')
        print('  📞 Doctor notification sent')
    else:
        print('  ✅ Dose on track — no action needed')

print('✅ All pipeline components defined')

✅ All pipeline components defined


---
## Cell 8 — Run the Full Agent Pipeline

Three scenarios demonstrating the full flow end-to-end.

In [9]:
def run_agent(sequence, adherence_pct, label=''):
    """Full pipeline: Input → LSTM → GPT-4 Agent Brain → Tool → Output"""
    print(f'\n{"="*55}')
    if label:
        print(f'  SCENARIO: {label}')
    print(f'  ① INPUT         Last 3 doses: {sequence}')
    print(f'                  Weekly adherence: {adherence_pct:.1f}%')

    prediction, confidence = lstm_predict(sequence)
    print(f'  ② LSTM MODEL    Predicts next dose: {prediction}  ({confidence:.1%} confidence)')

    print(f'  ③ GPT-4 BRAIN   Reasoning...')
    action, reason = agent_brain(sequence, prediction, confidence, adherence_pct)
    print(f'                  Action : {action}')
    print(f'                  Reason : {reason}')

    print(f'  ④ TOOL          Executing...')
    notification_tool(action)
    print(f'{"="*55}')


# ── SCENARIO 1: Patient has been missing recent doses ──
run_agent([1, 0, 0], adherence_pct=50.0, label='Patient missing recent doses')

# ── SCENARIO 2: Patient is consistently taking doses ──
run_agent([1, 1, 1], adherence_pct=92.0, label='Patient on track')

# ── SCENARIO 3: Chronic non-adherence — needs escalation ──
run_agent([0, 0, 0], adherence_pct=33.0, label='Severe non-adherence — escalate to doctor')


  SCENARIO: Patient missing recent doses
  ① INPUT         Last 3 doses: [1, 0, 0]
                  Weekly adherence: 50.0%
  ② LSTM MODEL    Predicts next dose: Missed  (47.9% confidence)
  ③ GPT-4 BRAIN   Reasoning...
                  Action : Send Reminder
                  Reason : The patient has missed the last two doses and the model predicts a possible miss for the next dose, despite the low confidence.
  ④ TOOL          Executing...
  🔔 ALERT: Sending medication reminder to patient
  📱 SMS dispatched to patient phone
  👩‍⚕️  Caregiver dashboard updated

  SCENARIO: Patient on track
  ① INPUT         Last 3 doses: [1, 1, 1]
                  Weekly adherence: 92.0%
  ② LSTM MODEL    Predicts next dose: Missed  (39.2% confidence)
  ③ GPT-4 BRAIN   Reasoning...
                  Action : No Action
                  Reason : The patient has been consistent with their medication recently and the model's prediction of a missed dose is low in confidence.
  ④ TOOL          Executin

---
## Cell 9 — Memory Inspection

Shows that the agent remembered all three interactions, building context over time.

In [10]:
print('=== AGENT MEMORY LOG ===')
print(f'Total messages in memory: {len(conversation_memory)}\n')
for i, msg in enumerate(conversation_memory):
    role = msg['role'].upper()
    preview = msg['content'][:120].replace('\n', ' ')
    print(f'[{i}] {role}: {preview}...')
    print()

=== AGENT MEMORY LOG ===
Total messages in memory: 7

[0] SYSTEM: You are a clinical medication adherence assistant. You receive a patient's dose history and an LSTM model prediction, th...

[1] USER: Patient data: - Last 3 doses: [1, 0, 0] - LSTM prediction for next dose: Missed (confidence: 47.9%) - Overall weekly adh...

[2] ASSISTANT: ACTION: Send Reminder REASON: The patient has missed the last two doses and the model predicts a possible miss for the n...

[3] USER: Patient data: - Last 3 doses: [1, 1, 1] - LSTM prediction for next dose: Missed (confidence: 39.2%) - Overall weekly adh...

[4] ASSISTANT: ACTION: No Action REASON: The patient has been consistent with their medication recently and the model's prediction of a...

[5] USER: Patient data: - Last 3 doses: [0, 0, 0] - LSTM prediction for next dose: Missed (confidence: 48.6%) - Overall weekly adh...

[6] ASSISTANT: ACTION: Escalate to Doctor REASON: The patient has missed the last three doses and has a low overall weekly a

---
## Cell 10 — Weekly Adherence Summary Report

In [11]:
print('\n' + '='*50)
print('    WEEKLY ADHERENCE SUMMARY REPORT')
print('='*50)

bar = ''.join(['█' if d == 1 else '░' for d in dose_history])
print(f'\n  Dose log:  {bar}')
print(f'             (█ = taken, ░ = missed)')
print(f'\n  Total doses scheduled : {total}')
print(f'  Doses taken           : {taken}')
print(f'  Doses missed          : {total - taken}')
print(f'  Adherence rate        : {pct:.1f}%')

print(f'\n  Risk level  : ', end='')
if pct >= 80:
    print('🟢 LOW — patient is adherent')
elif pct >= 60:
    print('🟡 MEDIUM — occasional misses detected')
else:
    print('🔴 HIGH — significant non-adherence')

last_3 = dose_history[-3:]
next_pred, next_conf = lstm_predict(last_3)
print(f'\n  Last 3 doses          : {last_3}')
print(f'  LSTM next prediction  : {next_pred} ({next_conf:.1%} confidence)')
print('\n' + '='*50)


    WEEKLY ADHERENCE SUMMARY REPORT

  Dose log:  ██░█░░██░█░░
             (█ = taken, ░ = missed)

  Total doses scheduled : 12
  Doses taken           : 6
  Doses missed          : 6
  Adherence rate        : 50.0%

  Risk level  : 🔴 HIGH — significant non-adherence

  Last 3 doses          : [1, 0, 0]
  LSTM next prediction  : Missed (47.9% confidence)



---
## Architecture Reference

```
┌──────────────────────────────────────────────────────────┐
│                     AGENT PIPELINE                       │
│                                                          │
│  ┌──────────┐   ┌────────────┐   ┌────────────────────┐  │
│  │  INPUT   │──▶│  DL MODEL  │──▶│   AGENT BRAIN      │  │
│  │          │   │  (LSTM)    │   │   (GPT-4 LLM)      │  │
│  │ 3 doses  │   │            │   │                    │  │
│  │ [1, 0, 0]│   │ predicts:  │   │ + MEMORY           │  │
│  │ adherence│   │ Taken /    │   │ (conversation      │  │
│  │ rate     │   │ Missed     │   │  history)          │  │
│  └──────────┘   └────────────┘   └────────┬───────────┘  │
│                                           │              │
│                                  ┌────────▼───────────┐  │
│                                  │       TOOL         │  │
│                                  │  notification_tool │  │
│                                  └────────┬───────────┘  │
│                                           │              │
│                                  ┌────────▼───────────┐  │
│                                  │      OUTPUT        │  │
│                                  │ 🔔 SMS Reminder    │  │
│                                  │ 🚨 Doctor Alert    │  │
│                                  │ ✅ No Action       │  │
│                                  └────────────────────┘  │
└──────────────────────────────────────────────────────────┘
```

### Deep Learning Concepts Applied
| Concept | Where Used |
|---|---|
| LSTM (Recurrent Neural Network) | Sequence prediction of dose adherence |
| Transformer / Attention | GPT-4 agent brain reasoning layer |
| Conversation Memory | Context-aware multi-turn decision making |
| Binary Classification | LSTM output: Taken vs Missed |